# ⚡ Fantasy World Predictions — Harry Potter Universe
## Step 3: ML Pipeline

---

### 🎯 Goal of this notebook
Same shared-factory pattern as `LOTR_ML_Pipeline.ipynb`: one-hot encode categoricals
(`handle_unknown='ignore'`), stratified 80/20 split, `class_weight='balanced'`, logreg vs
random forest per target (cloned estimators — not shared), model picked by **macro-F1**, save
the winner + a manifest for Streamlit.

**Targets:** `Gender`, `Species/Race`, `Blood`, `profession_group` — trained on the full
dataset. `hogwarts_house` is trained on the `is_hogwarts == 1` subset only (restricting to
Hogwarts-affiliated rows, since otherwise `"None"` dominates ~85% of rows).

---

## 📦 Section 1: Imports & Load

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.base import clone

os.makedirs('../models', exist_ok=True)

df = pd.read_csv('../data/hp_characters_clean.csv')
print(f'Loaded: {df.shape}')
df.head()

## ⚙️ Section 2: Shared factory

Column groups and the `train_target` factory function — identical structure to the LOTR
pipeline, just pointed at HP's columns.

In [ ]:
ALL_FEATURE_COLS = ['Gender', 'Species/Race', 'Blood', 'is_hogwarts', 'hogwarts_house',
                     'profession_group', 'has_description']
CATEGORICAL = {'Gender', 'Species/Race', 'Blood', 'hogwarts_house', 'profession_group'}
BINARY = {'is_hogwarts', 'has_description'}

def build_preprocessor(categorical_cols, binary_cols):
    transformers = []
    if categorical_cols:
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols))
    if binary_cols:
        transformers.append(('bin', 'passthrough', binary_cols))
    return ColumnTransformer(transformers)

def train_target(df_in, target, feature_cols, weak_target=False):
    data = df_in.dropna(subset=[target]).copy()
    X = data[feature_cols]
    y = data[target].astype(str)

    cat_cols = [c for c in feature_cols if c in CATEGORICAL]
    bin_cols = [c for c in feature_cols if c in BINARY]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pre = build_preprocessor(cat_cols, bin_cols)
    candidates = {
        'logreg': LogisticRegression(max_iter=1000, class_weight='balanced'),
        'random_forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    }

    best_name, best_pipe, best_f1, best_metrics = None, None, -1, None
    for name, est in candidates.items():
        pipe = Pipeline([('pre', pre), ('clf', clone(est))])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        f1m = f1_score(y_test, preds, average='macro')
        metrics = {
            'accuracy': accuracy_score(y_test, preds),
            'balanced_accuracy': balanced_accuracy_score(y_test, preds),
            'f1_macro': f1m,
            'f1_weighted': f1_score(y_test, preds, average='weighted'),
        }
        if f1m > best_f1:
            best_name, best_pipe, best_f1, best_metrics = name, pipe, f1m, metrics

    print(f'{target}: winner={best_name}  n_test={len(y_test)}  f1_macro={best_f1:.3f}')

    return {
        'pipeline': best_pipe,
        'feature_columns': feature_cols,
        'classes': sorted(y.unique().tolist()),
        'task': 'classification',
        'model_name': best_name,
        'metrics': best_metrics,
        'weak_target': weak_target,
    }

## 🏋️ Section 3: Train the full-dataset targets

`Gender`, `Species/Race`, `Blood`, `profession_group` — each uses every other engineered column
as a feature.

In [ ]:
manifest = {'universe': 'harry_potter', 'n_rows': len(df), 'id_column': 'Name', 'targets': {}}

targets_config = [
    ('Gender', df, [c for c in ALL_FEATURE_COLS if c != 'Gender'], False),
    ('Species/Race', df, [c for c in ALL_FEATURE_COLS if c != 'Species/Race'], False),
    ('Blood', df, [c for c in ALL_FEATURE_COLS if c != 'Blood'], True),
    ('profession_group', df, [c for c in ALL_FEATURE_COLS if c != 'profession_group'], True),
]

for target, data_in, feats, weak in targets_config:
    result = train_target(data_in, target, feats, weak)
    safe_name = target.lower().replace('/', '_')
    joblib.dump(result, f'../models/hp_{safe_name}.pkl')
    manifest['targets'][target] = {
        'task': result['task'],
        'weak_target': result['weak_target'],
        'model_file': f'hp_{safe_name}.pkl',
        'model_name': result['model_name'],
        'feature_columns': result['feature_columns'],
        'classes': result['classes'],
        'metrics': result['metrics'],
    }

## 🏰 Section 4: Train `hogwarts_house` (Hogwarts-only subset)

Restricted to `is_hogwarts == 1` rows so the model actually learns to distinguish the 4 real
houses, instead of mostly predicting `"None"`. `is_hogwarts` is dropped from the feature set
here since it's constant (always 1) within this subset.

In [ ]:
hog_df = df[df['is_hogwarts'] == 1].copy()
hog_feats = ['Gender', 'Species/Race', 'Blood', 'profession_group', 'has_description']

print(f'Hogwarts-only subset: {len(hog_df)} rows')

result = train_target(hog_df, 'hogwarts_house', hog_feats, weak_target=True)
joblib.dump(result, '../models/hp_hogwarts_house.pkl')
manifest['targets']['hogwarts_house'] = {
    'task': result['task'],
    'weak_target': result['weak_target'],
    'model_file': 'hp_hogwarts_house.pkl',
    'model_name': result['model_name'],
    'feature_columns': result['feature_columns'],
    'classes': result['classes'],
    'metrics': result['metrics'],
    'note': f'trained on is_hogwarts==1 subset only (n={len(hog_df)})',
}

## 💾 Section 5: Save manifest

In [ ]:
with open('../models/hp_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('Saved models:', [f for f in os.listdir('../models') if f.startswith('hp_')])

## 📊 Section 6: Test metrics summary

In [ ]:
rows = []
for target, info in manifest['targets'].items():
    rows.append({
        'target': target,
        'winner': info['model_name'],
        'accuracy': round(info['metrics']['accuracy'], 3),
        'f1_macro': round(info['metrics']['f1_macro'], 3),
        'weak_target': info['weak_target'],
    })
pd.DataFrame(rows).sort_values('f1_macro', ascending=False)

## 🔍 Section 7: Smoke test

Reload `hp_gender.pkl` from disk and predict one hand-built row — confirms the saved pickle is
self-contained and actually loadable outside this notebook, the same check we ran for LOTR.

In [ ]:
obj = joblib.load('../models/hp_gender.pkl')
pipe = obj['pipeline']
cols = obj['feature_columns']

sample = pd.DataFrame([{
    'Species/Race': 'wizard',
    'Blood': 'Pure Blood',
    'is_hogwarts': 1,
    'hogwarts_house': 'Gryffindor',
    'profession_group': 'Ministry',
    'has_description': 1,
}])[cols]

print('Prediction:', pipe.predict(sample))

---
## 🔜 What's Next?

Harry Potter is now at parity with LOTR: cleaned data + saved models + manifest. Reality check
on these numbers before moving on — some are noticeably weaker than LOTR's, which reflects the
data itself (fewer usable features, sparser labels), not a pipeline problem:

- `Gender` — strong (this dataset's cleanest column, as expected)
- `Species/Race`, `Blood` — moderate/weak, limited by how few real signals the remaining columns carry
- `profession_group`, `hogwarts_house` — weak, small/imbalanced classes, flagged as weak targets

Next natural step: **Star Wars EDA** (`03_starwars_eda_cleaning.ipynb`) — smallest dataset (~87
rows) so it should move fast, then Game of Thrones, then the Streamlit app tying all four
universes together.

> **Tell your assistant:** *"HP pipeline looks good, let's move to Star Wars EDA."* (or flag
> anything you want tuned first.)
